# 02 - Grafové algoritmy v Pythonu

Grafy jsou základní datové struktury, které modelují vztahy mezi objekty. Skládají se z množiny **vrcholů** ($V$, uzly) a množiny **hran** ($E$, spojení mezi uzly).

* **Orientovaný vs. neorientovaný graf:** Orientovaná hrana vede jen jedním směrem (`u → v`), neorientovaná oběma směry.
* **Ohodnocený vs. neohodnocený graf:** Ohodnocené hrany nesou číselnou váhu (vzdálenost, čas, cenu).

V tomto notebooku si ukážeme:
1. **Reprezentaci grafu** pomocí seznamu sousednosti (`dict` / `defaultdict`) a jeho **vizualizaci** (`networkx` + `matplotlib`)
2. **BFS (Breadth-First Search)** – prohledávání do šířky pomocí `collections.deque`
3. **DFS (Depth-First Search)** – prohledávání do hloubky (rekurzivně)
4. **Dijkstrův algoritmus** – nejkratší cesta v ohodnoceném grafu pomocí `heapq`
5. **Interaktivní krokování** všech tří algoritmů přímo v buňce

## 1. Reprezentace grafu a vizualizace

V paměti počítače reprezentujeme graf nejčastěji dvěma způsoby:
* **Matice sousednosti:** 2D pole $V \times V$. Zjištění hrany trvá $O(1)$, ale zabírá vždy $O(V^2)$ paměti.
* **Seznam sousednosti (Adjacency List):** Každý vrchol má seznam svých odchozích hran (cíl a volitelně váha). Zabírá jen $O(V + E)$ paměti, což je ideální pro řídké grafy.

Vytvoříme si datovou třídu `Hrana`, třídu `Graf` a pomocnou funkci `vykresli_graf` pro přehledné vykreslení grafu.

In [ ]:
from collections import defaultdict, deque
from dataclasses import dataclass
import heapq
import math

import matplotlib.pyplot as plt
import networkx as nx


@dataclass
class Hrana:
    cil: int
    vaha: int = 1


class Graf:
    def __init__(self, orientovany: bool = True) -> None:
        self.orientovany = orientovany
        self.seznam_sousedu: dict[int, list[Hrana]] = defaultdict(list)

    def pridej_hranu(self, od: int, do_vrcholu: int, vaha: int = 1) -> None:
        self.seznam_sousedu[od].append(Hrana(do_vrcholu, vaha))
        if not self.orientovany:
            self.seznam_sousedu[do_vrcholu].append(Hrana(od, vaha))


POZICE = {
    0: (0, 2),
    1: (1.5, 2.5),
    2: (1.5, 1.5),
    3: (3, 2),
    4: (2.2, 0.5),
    5: (4.5, 2),
}


def vykresli_graf(
    graf: Graf,
    titulek: str = "Ukázkový graf",
    zvyraznena_cesta: list[int] | None = None,
    poradi_navstevy: list[int] | None = None,
) -> None:
    G = nx.DiGraph() if graf.orientovany else nx.Graph()
    for u, hrany in graf.seznam_sousedu.items():
        for h in hrany:
            G.add_edge(u, h.cil, weight=h.vaha)

    zvyraznene_hrany = set()
    zvyraznene_uzly = set()
    if zvyraznena_cesta:
        zvyraznene_uzly = set(zvyraznena_cesta)
        zvyraznene_hrany = set(zip(zvyraznena_cesta, zvyraznena_cesta[1:]))

    barvy_uzlu = [
        "#4CAF50" if u in zvyraznene_uzly else "#90CAF9"
        for u in G.nodes()
    ]
    barvy_hran = [
        "#E53935" if (u, v) in zvyraznene_hrany else "#607D8B"
        for u, v in G.edges()
    ]
    sirky_hran = [
        3.0 if (u, v) in zvyraznene_hrany else 1.5
        for u, v in G.edges()
    ]

    popisky_uzlu = {}
    if poradi_navstevy:
        krok = {u: idx + 1 for idx, u in enumerate(poradi_navstevy)}
        popisky_uzlu = {u: f"{u}\n(#{krok[u]})" if u in krok else str(u) for u in G.nodes()}
    else:
        popisky_uzlu = {u: str(u) for u in G.nodes()}

    plt.figure(figsize=(8, 4.2))
    nx.draw(
        G,
        pos=POZICE,
        labels=popisky_uzlu,
        with_labels=True,
        node_color=barvy_uzlu,
        node_size=1400,
        font_size=10,
        font_weight="bold",
        edge_color=barvy_hran,
        width=sirky_hran,
        arrowsize=18,
    )
    vahy = nx.get_edge_attributes(G, "weight")
    nx.draw_networkx_edge_labels(G, pos=POZICE, edge_labels=vahy, font_size=10)
    plt.title(titulek, fontsize=13, fontweight="bold")
    plt.axis("off")
    plt.tight_layout()
    plt.show()


g = Graf(orientovany=True)
g.pridej_hranu(0, 1, 4)
g.pridej_hranu(0, 2, 2)
g.pridej_hranu(1, 3, 1)
g.pridej_hranu(2, 3, 3)
g.pridej_hranu(2, 4, 5)
g.pridej_hranu(4, 3, 1)
g.pridej_hranu(3, 5, 2)

vykresli_graf(g, titulek="Ukázkový orientovaný ohodnocený graf")

## 2. BFS (Breadth-First Search) – Prohledávání do šířky

BFS prohledává graf po „vrstvách“ podle počtu hran od startovního vrcholu:
1. Vložíme `start` do **fronty (`collections.deque`)** a ihned ho označíme jako navštívený.
2. Dokud fronta není prázdná, odebereme vrchol ze začátku (`popleft()`, čas $O(1)$) a přidáme na konec všechny jeho dosud nenavštívené sousedy (které zároveň označíme jako navštívené, abychom je nepřidali vícekrát).

* **Složitost:** Časová $O(V + E)$, paměťová $O(V)$.
* **Využití:** Díky průchodu po vrstvách **zaručuje nalezení nejkratší cesty** v neohodnoceném grafu (nejmenší počet hran).

Všimněte si na grafu níže pořadí návštěvy vrcholů `(#1, #2, ...)`: nejprve vrchol `0`, pak jeho sousedé `1, 2`, poté `3, 4` a nakonec `5`.

In [ ]:
def bfs(graf: Graf, start: int) -> list[int]:
    navstiveno: set[int] = {start}
    fronta: deque[int] = deque([start])
    poradi: list[int] = []

    while fronta:
        aktualni = fronta.popleft()
        poradi.append(aktualni)

        for hrana in graf.seznam_sousedu[aktualni]:
            if hrana.cil not in navstiveno:
                navstiveno.add(hrana.cil)
                fronta.append(hrana.cil)
    return poradi


poradi_bfs = bfs(g, 0)
print("BFS průchod od 0:", poradi_bfs)
vykresli_graf(g, titulek=f"BFS průchod: {poradi_bfs}", poradi_navstevy=poradi_bfs)

## 3. DFS (Depth-First Search) – Prohledávání do hloubky

DFS jde vždy co nejdále po jedné větvi (`0 → 1 → 3 → 5`) a při slepé uličce nebo již navštíveném vrcholu se vrací (*backtracking*) na poslední křižovatku a projde zbylou větev (`2 → 4`). Lze ho napsat **rekurzivně** nebo pomocí **zásobníku (LIFO)**.

* **Složitost:** Časová $O(V + E)$, paměťová $O(V)$.
* **Vlastnosti:** DFS **nezaručuje** nalezení nejkratší cesty.
* **Využití:** Hledání cyklů, topologické řazení (závislosti), komponenty souvislosti, prohledávání bludišť a generování všech cest.

In [ ]:
def dfs_rekurzivne(graf: Graf, start: int, navstiveno: set[int] | None = None) -> list[int]:
    if navstiveno is None:
        navstiveno = set()
    navstiveno.add(start)
    poradi = [start]

    for hrana in graf.seznam_sousedu[start]:
        if hrana.cil not in navstiveno:
            poradi.extend(dfs_rekurzivne(graf, hrana.cil, navstiveno))
    return poradi


poradi_dfs = dfs_rekurzivne(g, 0)
print("DFS průchod od 0:", poradi_dfs)
vykresli_graf(g, titulek=f"DFS průchod: {poradi_dfs}", poradi_navstevy=poradi_dfs)

## 4. Dijkstrův algoritmus – Nejkratší cesta v ohodnoceném grafu

Pokud mají hrany **nezáporné váhy** (vzdálenost, čas, cena), BFS nestačí, protože nejkratší cesta může mít více hran, ale menší součet vah. Dijkstrův algoritmus:
1. Nastaví počáteční vzdálenosti na `∞` (pro `start` na `0`).
2. Pomocí **prioritní fronty (`heapq`)** vždy vybere dosud neuzavřený vrchol `u` s nejmenší vzdáleností.
3. Provede **relaxaci hran** z `u`: pokud je cesta přes `u` k sousedovi `v` kratší (`akt_vzd + w < vzd[v]`), aktualizuje vzdálenost, uloží předchůdce `pred[v] = u` a vloží `(nova_vzd, v)` do haldy.

* **Podmínka:** Hrany nesmí mít záporné váhy.
* **Složitost:** Časová $O(E \log V)$, paměťová $O(V + E)$.

In [ ]:
def dijkstra(graf: Graf, start: int, cil: int) -> tuple[int, list[int]] | None:
    vzdalenosti: dict[int, float] = {v: math.inf for v in graf.seznam_sousedu}
    for hrany in graf.seznam_sousedu.values():
        for hrana in hrany:
            vzdalenosti.setdefault(hrana.cil, math.inf)
    vzdalenosti[start] = 0

    predchudci: dict[int, int] = {}
    pq: list[tuple[float, int]] = [(0, start)]

    while pq:
        akt_vzd, akt_vrchol = heapq.heappop(pq)
        if akt_vzd > vzdalenosti[akt_vrchol]:
            continue
        if akt_vrchol == cil:
            break

        for hrana in graf.seznam_sousedu[akt_vrchol]:
            nova_vzd = akt_vzd + hrana.vaha
            if nova_vzd < vzdalenosti[hrana.cil]:
                vzdalenosti[hrana.cil] = nova_vzd
                predchudci[hrana.cil] = akt_vrchol
                heapq.heappush(pq, (nova_vzd, hrana.cil))

    if vzdalenosti.get(cil, math.inf) == math.inf:
        return None

    cesta: list[int] = []
    v: int | None = cil
    while v is not None:
        cesta.append(v)
        if v == start:
            break
        v = predchudci.get(v)
    cesta.reverse()
    return int(vzdalenosti[cil]), cesta


vysledek = dijkstra(g, 0, 5)
print("Nejkratší cesta (vzdálenost, trasa):", vysledek)
if vysledek is not None:
    cena, cesta = vysledek
    vykresli_graf(g, titulek=f"Dijkstra (cena = {cena}): {' -> '.join(map(str, cesta))}", zvyraznena_cesta=cesta)

## 5. Interaktivní krokování algoritmů (BFS, DFS, Dijkstra)

Spusťte buňku níže a vyzkoušejte si **krok za krokem** (`▶ Přehrát` nebo `Vpřed ▶`), jak jednotlivé algoritmy procházejí grafem a jak se mění **Fronta (BFS)**, **Zásobník (DFS)** a **Prioritní fronta + vzdálenosti (Dijkstra)**.

In [ ]:
import json
from IPython.display import HTML, display


def trasuj_bfs(graf: Graf, start: int) -> list[dict]:
    kroky = []
    navstiveno: list[int] = [start]
    fronta: deque[int] = deque([start])
    uzavreno: list[int] = []
    strom_hran: list[list[int]] = []

    kroky.append({
        "aktualni": None,
        "ve_fronte": list(fronta),
        "uzavreno": list(uzavreno),
        "aktivni_hrana": None,
        "strom_hran": list(strom_hran),
        "struktura": f"Fronta: {list(fronta)}",
        "popis": f"Start BFS z vrcholu {start}. Vložen do fronty.",
    })

    while fronta:
        akt = fronta.popleft()
        uzavreno.append(akt)
        kroky.append({
            "aktualni": akt,
            "ve_fronte": list(fronta),
            "uzavreno": list(uzavreno),
            "aktivni_hrana": None,
            "strom_hran": list(strom_hran),
            "struktura": f"Fronta: {list(fronta)}",
            "popis": f"Odebrán vrchol {akt} ze začátku fronty.",
        })
        for h in graf.seznam_sousedu[akt]:
            if h.cil not in navstiveno:
                navstiveno.append(h.cil)
                fronta.append(h.cil)
                strom_hran.append([akt, h.cil])
                kroky.append({
                    "aktualni": akt,
                    "ve_fronte": list(fronta),
                    "uzavreno": list(uzavreno),
                    "aktivni_hrana": [akt, h.cil],
                    "strom_hran": list(strom_hran),
                    "struktura": f"Fronta: {list(fronta)}",
                    "popis": f"Soused {h.cil} ještě nebyl navštíven -> přidán do fronty.",
                })
    kroky.append({
        "aktualni": None,
        "ve_fronte": [],
        "uzavreno": list(uzavreno),
        "aktivni_hrana": None,
        "strom_hran": list(strom_hran),
        "struktura": "Fronta: []",
        "popis": f"Hotovo! Pořadí BFS: {uzavreno}",
    })
    return kroky


def trasuj_dfs(graf: Graf, start: int) -> list[dict]:
    kroky = []
    navstiveno: list[int] = []
    zasobnik: list[int] = [start]

    kroky.append({
        "aktualni": None,
        "ve_fronte": list(zasobnik),
        "uzavreno": list(navstiveno),
        "aktivni_hrana": None,
        "strom_hran": [],
        "struktura": f"Zásobník: {list(zasobnik)}",
        "popis": f"Start DFS z vrcholu {start}. Vložen na zásobník.",
    })

    strom_hran: list[list[int]] = []
    rodic: dict[int, int] = {}
    while zasobnik:
        akt = zasobnik.pop()
        if akt in navstiveno:
            continue
        navstiveno.append(akt)
        if akt in rodic:
            strom_hran.append([rodic[akt], akt])
        kroky.append({
            "aktualni": akt,
            "ve_fronte": list(zasobnik),
            "uzavreno": list(navstiveno),
            "aktivni_hrana": [rodic[akt], akt] if akt in rodic else None,
            "strom_hran": list(strom_hran),
            "struktura": f"Zásobník: {list(zasobnik)}",
            "popis": f"Zpracován vrchol {akt} z vrcholu zásobníku.",
        })
        for h in reversed(graf.seznam_sousedu[akt]):
            if h.cil not in navstiveno:
                zasobnik.append(h.cil)
                rodic[h.cil] = akt
                kroky.append({
                    "aktualni": akt,
                    "ve_fronte": list(zasobnik),
                    "uzavreno": list(navstiveno),
                    "aktivni_hrana": [akt, h.cil],
                    "strom_hran": list(strom_hran),
                    "struktura": f"Zásobník: {list(zasobnik)}",
                    "popis": f"Soused {h.cil} vložen na zásobník.",
                })
    kroky.append({
        "aktualni": None,
        "ve_fronte": [],
        "uzavreno": list(navstiveno),
        "aktivni_hrana": None,
        "strom_hran": list(strom_hran),
        "struktura": "Zásobník: []",
        "popis": f"Hotovo! Pořadí DFS: {navstiveno}",
    })
    return kroky


def trasuj_dijkstra(graf: Graf, start: int, cil: int) -> list[dict]:
    kroky = []
    vzd = {v: math.inf for v in POZICE}
    vzd[start] = 0
    pred: dict[int, int] = {}
    pq: list[tuple[float, int]] = [(0, start)]
    uzavreno: list[int] = []

    def fmt_vzd():
        return {str(k): ("∞" if val == math.inf else int(val)) for k, val in sorted(vzd.items())}

    kroky.append({
        "aktualni": None,
        "ve_fronte": [start],
        "uzavreno": [],
        "aktivni_hrana": None,
        "strom_hran": [],
        "struktura": f"PQ: [(0, {start})] | Vzdálenosti: {fmt_vzd()}",
        "popis": f"Start Dijkstry z {start} do {cil}. Vzdálenost({start}) = 0.",
    })

    while pq:
        d, u = heapq.heappop(pq)
        if d > vzd[u]:
            continue
        uzavreno.append(u)
        kroky.append({
            "aktualni": u,
            "ve_fronte": [n for _, n in pq],
            "uzavreno": list(uzavreno),
            "aktivni_hrana": None,
            "strom_hran": [],
            "struktura": f"PQ: {[(int(x), y) for x, y in pq]} | Vzdálenosti: {fmt_vzd()}",
            "popis": f"Vybrán nejbližší vrchol {u} (vzdálenost {int(d)}).",
        })
        if u == cil:
            break
        for h in graf.seznam_sousedu[u]:
            nova = d + h.vaha
            if nova < vzd[h.cil]:
                vzd[h.cil] = nova
                pred[h.cil] = u
                heapq.heappush(pq, (nova, h.cil))
                kroky.append({
                    "aktualni": u,
                    "ve_fronte": [n for _, n in pq],
                    "uzavreno": list(uzavreno),
                    "aktivni_hrana": [u, h.cil],
                    "strom_hran": [],
                    "struktura": f"PQ: {[(int(x), y) for x, y in pq]} | Vzdálenosti: {fmt_vzd()}",
                    "popis": f"Relaxace hrany {u} -> {h.cil} (váha {h.vaha}): nová vzdálenost = {int(nova)}.",
                })

    cesta: list[int] = []
    curr: int | None = cil
    while curr is not None:
        cesta.append(curr)
        if curr == start:
            break
        curr = pred.get(curr)
    cesta.reverse()
    cesta_hrany = [[a, b] for a, b in zip(cesta, cesta[1:])]
    kroky.append({
        "aktualni": None,
        "ve_fronte": [],
        "uzavreno": list(uzavreno),
        "aktivni_hrana": None,
        "strom_hran": cesta_hrany,
        "struktura": f"Vzdálenosti: {fmt_vzd()}",
        "popis": f"Hotovo! Nejkratší cesta {start} -> {cil} má cenu {int(vzd[cil])}: {' -> '.join(map(str, cesta))}",
    })
    return kroky


def zobraz_interaktivni_graf(graf: Graf) -> None:
    data = {
        "uzly": {str(k): [v[0] * 120 + 60, (3 - v[1]) * 95 + 30] for k, v in POZICE.items()},
        "hrany": [
            {"u": u, "v": h.cil, "w": h.vaha}
            for u, hrany in graf.seznam_sousedu.items()
            for h in hrany
        ],
        "algoritmy": {
            "BFS": trasuj_bfs(graf, 0),
            "DFS": trasuj_dfs(graf, 0),
            "Dijkstra": trasuj_dijkstra(graf, 0, 5),
        },
    }
    data_json = json.dumps(data, ensure_ascii=False)
    html = f"""
    <div id="gv-app" style="font-family:system-ui,sans-serif;border:1px solid #d0d7de;border-radius:10px;padding:14px;background:#fff;color:#1f2328;max-width:860px;">
      <div style="display:flex;gap:8px;flex-wrap:wrap;align-items:center;margin-bottom:10px;">
        <strong style="margin-right:6px;">Algoritmus:</strong>
        <button class="gv-alg" data-alg="BFS" style="padding:5px 12px;border-radius:6px;border:1px solid #0969da;background:#0969da;color:#fff;cursor:pointer;">BFS (Fronta)</button>
        <button class="gv-alg" data-alg="DFS" style="padding:5px 12px;border-radius:6px;border:1px solid #d0d7de;background:#f6f8fa;cursor:pointer;">DFS (Zásobník)</button>
        <button class="gv-alg" data-alg="Dijkstra" style="padding:5px 12px;border-radius:6px;border:1px solid #d0d7de;background:#f6f8fa;cursor:pointer;">Dijkstra (PQ)</button>
      </div>
      <div style="display:flex;gap:8px;flex-wrap:wrap;align-items:center;margin-bottom:10px;">
        <button id="gv-reset" style="padding:4px 10px;border-radius:6px;border:1px solid #d0d7de;background:#f6f8fa;cursor:pointer;">⏮ Reset</button>
        <button id="gv-prev" style="padding:4px 10px;border-radius:6px;border:1px solid #d0d7de;background:#f6f8fa;cursor:pointer;">◀ Zpět</button>
        <button id="gv-play" style="padding:4px 10px;border-radius:6px;border:1px solid #1a7f37;background:#1a7f37;color:#fff;cursor:pointer;">▶ Přehrát</button>
        <button id="gv-next" style="padding:4px 10px;border-radius:6px;border:1px solid #d0d7de;background:#f6f8fa;cursor:pointer;">Vpřed ▶</button>
        <input id="gv-slider" type="range" min="0" max="0" value="0" style="flex:1;min-width:140px;">
        <span id="gv-counter" style="font-size:13px;font-weight:600;">Krok 0 / 0</span>
      </div>
      <svg id="gv-svg" viewBox="0 0 660 310" style="width:100%;height:auto;border:1px solid #eaeef2;border-radius:8px;background:#fafbfc;">
        <defs>
          <marker id="arrow" viewBox="0 0 10 10" refX="25" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
            <path d="M 0 1 L 10 5 L 0 9 z" fill="#6e7781"/>
          </marker>
          <marker id="arrow-active" viewBox="0 0 10 10" refX="25" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
            <path d="M 0 1 L 10 5 L 0 9 z" fill="#cf222e"/>
          </marker>
          <marker id="arrow-tree" viewBox="0 0 10 10" refX="25" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
            <path d="M 0 1 L 10 5 L 0 9 z" fill="#1a7f37"/>
          </marker>
        </defs>
        <g id="gv-edges"></g>
        <g id="gv-nodes"></g>
      </svg>
      <div style="margin-top:10px;padding:10px;border-radius:8px;background:#f6f8fa;font-size:13px;line-height:1.5;">
        <div><strong>Krok:</strong> <span id="gv-desc"></span></div>
        <div><strong>Stav struktury:</strong> <code id="gv-struct"></code></div>
        <div><strong>Navštíveno:</strong> <code id="gv-visited"></code></div>
        <div style="margin-top:4px;font-size:12px;color:#57606a;">Legenda: <span style="color:#0969da;">● Nenavštíveno</span> &nbsp; <span style="color:#bf8700;">● Ve frontě/zásobníku</span> &nbsp; <span style="color:#cf222e;">● Aktuální</span> &nbsp; <span style="color:#1a7f37;">● Zpracováno</span></div>
      </div>
    </div>
    <script>
    (function() {{
      const DATA = {data_json};
      let alg = 'BFS';
      let idx = 0;
      let timer = null;
      const svgEdges = document.getElementById('gv-edges');
      const svgNodes = document.getElementById('gv-nodes');
      const slider = document.getElementById('gv-slider');
      const counter = document.getElementById('gv-counter');
      const desc = document.getElementById('gv-desc');
      const structEl = document.getElementById('gv-struct');
      const visitedEl = document.getElementById('gv-visited');
      const playBtn = document.getElementById('gv-play');

      function stopPlay() {{
        if (timer) {{ clearInterval(timer); timer = null; playBtn.textContent = '▶ Přehrát'; }}
      }}

      function render() {{
        const kroky = DATA.algoritmy[alg];
        const s = kroky[idx];
        slider.max = kroky.length - 1;
        slider.value = idx;
        counter.textContent = `Krok ${{idx + 1}} / ${{kroky.length}}`;
        desc.textContent = s.popis;
        structEl.textContent = s.struktura;
        visitedEl.textContent = JSON.stringify(s.uzavreno);

        svgEdges.innerHTML = '';
        for (const e of DATA.hrany) {{
          const p1 = DATA.uzly[String(e.u)];
          const p2 = DATA.uzly[String(e.v)];
          const isActive = s.aktivni_hrana && s.aktivni_hrana[0] === e.u && s.aktivni_hrana[1] === e.v;
          const isTree = s.strom_hran.some(h => h[0] === e.u && h[1] === e.v);
          const color = isActive ? '#cf222e' : (isTree ? '#1a7f37' : '#8c959f');
          const width = (isActive || isTree) ? 3.2 : 1.8;
          const marker = isActive ? 'url(#arrow-active)' : (isTree ? 'url(#arrow-tree)' : 'url(#arrow)');
          const line = document.createElementNS('http://www.w3.org/2000/svg', 'line');
          line.setAttribute('x1', p1[0]); line.setAttribute('y1', p1[1]);
          line.setAttribute('x2', p2[0]); line.setAttribute('y2', p2[1]);
          line.setAttribute('stroke', color); line.setAttribute('stroke-width', width);
          line.setAttribute('marker-end', marker);
          svgEdges.appendChild(line);

          const mx = (p1[0] + p2[0]) / 2;
          const my = (p1[1] + p2[1]) / 2 - 6;
          const lbl = document.createElementNS('http://www.w3.org/2000/svg', 'text');
          lbl.setAttribute('x', mx); lbl.setAttribute('y', my);
          lbl.setAttribute('text-anchor', 'middle');
          lbl.setAttribute('font-size', '12'); lbl.setAttribute('font-weight', '600');
          lbl.setAttribute('fill', isActive ? '#cf222e' : '#24292f');
          lbl.textContent = e.w;
          svgEdges.appendChild(lbl);
        }}

        svgNodes.innerHTML = '';
        for (const [id, pos] of Object.entries(DATA.uzly)) {{
          const nid = Number(id);
          let fill = '#ddf4ff', stroke = '#0969da';
          if (s.uzavreno.includes(nid)) {{ fill = '#dafbe1'; stroke = '#1a7f37'; }}
          if (s.ve_fronte.includes(nid)) {{ fill = '#fff8c5'; stroke = '#bf8700'; }}
          if (s.aktualni === nid) {{ fill = '#ffebe9'; stroke = '#cf222e'; }}

          const c = document.createElementNS('http://www.w3.org/2000/svg', 'circle');
          c.setAttribute('cx', pos[0]); c.setAttribute('cy', pos[1]); c.setAttribute('r', 20);
          c.setAttribute('fill', fill); c.setAttribute('stroke', stroke);
          c.setAttribute('stroke-width', s.aktualni === nid ? 4 : 2.5);
          svgNodes.appendChild(c);

          const t = document.createElementNS('http://www.w3.org/2000/svg', 'text');
          t.setAttribute('x', pos[0]); t.setAttribute('y', pos[1] + 5);
          t.setAttribute('text-anchor', 'middle'); t.setAttribute('font-size', '15');
          t.setAttribute('font-weight', '700'); t.setAttribute('fill', '#1f2328');
          t.textContent = id;
          svgNodes.appendChild(t);
        }}
      }}

      document.querySelectorAll('.gv-alg').forEach(btn => {{
        btn.addEventListener('click', () => {{
          stopPlay();
          alg = btn.dataset.alg;
          idx = 0;
          document.querySelectorAll('.gv-alg').forEach(b => {{
            const active = b.dataset.alg === alg;
            b.style.background = active ? '#0969da' : '#f6f8fa';
            b.style.color = active ? '#fff' : '#1f2328';
            b.style.borderColor = active ? '#0969da' : '#d0d7de';
          }});
          render();
        }});
      }});

      document.getElementById('gv-reset').addEventListener('click', () => {{ stopPlay(); idx = 0; render(); }});
      document.getElementById('gv-prev').addEventListener('click', () => {{ stopPlay(); if (idx > 0) {{ idx--; render(); }} }});
      document.getElementById('gv-next').addEventListener('click', () => {{ stopPlay(); if (idx < DATA.algoritmy[alg].length - 1) {{ idx++; render(); }} }});
      slider.addEventListener('input', () => {{ stopPlay(); idx = Number(slider.value); render(); }});
      playBtn.addEventListener('click', () => {{
        if (timer) {{ stopPlay(); return; }}
        playBtn.textContent = '⏸ Pauza';
        timer = setInterval(() => {{
          if (idx < DATA.algoritmy[alg].length - 1) {{ idx++; render(); }}
          else {{ stopPlay(); }}
        }}, 900);
      }});

      render();
    }})();
    </script>
    """
    display(HTML(html))


zobraz_interaktivni_graf(g)

## 6. Cvičení k procvičení

1. Upravte funkci `bfs` tak, aby přijímala i `cil` a vracela konkrétní nejkratší cestu (seznam vrcholů), a vykreslete ji pomocí `vykresli_graf(..., zvyraznena_cesta=cesta)`.
2. Napište pomocí DFS funkci `je_dosazitelny(graf, start, cil) -> bool`.
3. Vytvořte vlastní neorientovaný graf s městy a vzdálenostmi v km a najděte mezi nimi nejkratší trasu pomocí Dijkstrova algoritmu.

In [ ]:
# Zde si můžete vyzkoušet řešení cvičení
